# Visual Asset Auditing System - Test Bench

This notebook implements the **Test Bench** tier of the Visual Asset Auditing System. It demonstrates:

1. Connecting to AlloyDB and GCS.
2. Running a Hybrid Search (pgvector + FTS).
3. Detecting the drop-off point and selecting the top 60 candidates (High Confidence + Borderline).
4. Running Gemini 3.5 Flash online inference to audit the selected assets.
5. Creating and updating the `audit_results` table in AlloyDB.

*Transcribed from IMG_9422.jpeg. Additional source screenshots will be added below in the order received.*

In [ ]:
# Install required libraries
!pip install -q google-genai google-cloud-vision google-cloud-aiplatform kfp google-cloud-pipeline-components pgvector asyncpg kneed pandas numpy pillow nest-asyncio sqlalchemy "protobuf<5.0.0dev"


## Package Installation

This cell installs all necessary external libraries (Google GenAI, Cloud Vision, AI Platform, pgvector, asyncio) to equip the notebook environment.

In [ ]:
# Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()

import google.genai as genai
from google.genai import types

print("Google GenAI SDK imported successfully.")


## Google Cloud Authentication

This cell handles authentication with Google Cloud using Colab auth utilities, sets up API project client.

In [ ]:
# AlloyDB Connection Setup using SQLAlchemy Pool + AsyncConnector
import asyncio
import asyncpg
from typing import Tuple
from sqlalchemy.ext.asyncio import create_async_engine, AsyncEngine
from google.cloud.alloydb.connector import IPTypes, AsyncConnector

_engine_cache = {}
_connector_cache = {}

async def get_alloydb_connection(reuse: bool = True) -> Tuple[AsyncEngine, AsyncConnector]:
    """Establishes and pools AlloyDB connections using SQLAlchemy and the AsyncConnector, with automatic timeout diagnostics."""
    global _engine_cache
    global _connector_cache

    if reuse and 'default' in _engine_cache:
        return _engine_cache['default'], _connector_cache['default']

    # Use lazy refresh for serverless/Colab environments
    connector = AsyncConnector(refresh_strategy="lazy")

    async def getconn():
        # Handle case where user pasted the full resource path or just the instance ID
        instance_uri = ALLOYDB_INSTANCE
        if not instance_uri.startswith("projects/"):
            instance_uri = f"projects/{PROJECT_ID}/locations/{REGION}/clusters/{ALLOYDB_CLUSTER}/instances/{ALLOYDB_INSTANCE}"

        try:
            # Enforce 10-second connection timeout to prevent hanging loop CancelledErrors
            conn = await asyncio.wait_for(
                connector.connect(
                    instance_uri,
                    "asyncpg",
                    user=DB_USER,
                    password=DB_PASSWORD,
                    db=DB_NAME,
                    enable_iam_auth=False, # Set to True if using IAM auth
                    ip_type=IPTypes.PUBLIC # Adjust to PUBLIC or PRIVATE
                ),
                timeout=10.0
            )
            return conn
        except asyncio.TimeoutError:
            raise ConnectionError(
                f"AlloyDB connection timed out (10s) to {instance_uri}. "
                "Ensure your client IP is authorized in the AlloyDB Public IP console, "
                "or check your VPC network access if running internally."
            )
        except Exception as e:
            raise ConnectionError(f"Failed to connect to AlloyDB: {e}")

    engine = create_async_engine(
        "postgresql+asyncpg://",
        async_creator=getconn,
        echo=False,
        pool_size=10,
        max_overflow=20,
        pool_pre_ping=True # Force SQLAlchemy to health-check connections
    )

    if reuse:
        _engine_cache['default'] = engine
        _connector_cache['default'] = connector

    return engine, connector


In [ ]:
# # Run the setup
# import nest_asyncio
# nest_asyncio.apply()
# try:
#     asyncio.run(setup_database())
# except Exception as e:
#     print(f"Skipping execution: Database connection not configured yet ({e})")


In [ ]:
# Database Connection Verification (Read-Only Check – No DDL or Schema Changes)
import asyncio
import nest_asyncio

async def verify_database_connection():
    """Verifies connection to AlloyDB and confirms the visual_assets table is reachable without making any DDL changes."""
    try:
        engine, _ = await get_alloydb_connection()
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            count = await db.fetchval(f"SELECT COUNT(*) FROM {DB_SCHEMA}.visual_assets;")
            print(f"Connected to AlloyDB successfully. Schema '{DB_SCHEMA}.visual_assets' is ready ({count} assets found).")
    except Exception as e:
        print(f"Database connection status: {e}")

# Run connection check
nest_asyncio.apply()
try:
    asyncio.run(verify_database_connection())
except Exception as e:
    print(f"Skipping execution: Database connection not configured yet ({e})")


In [ ]:
# 1. Audit Config Generator & Embedding Generation
import json
import asyncio
from concurrent.futures import ThreadPoolExecutor
from typing import Optional, List, Dict, Tuple
from pydantic import BaseModel, Field

# Define Pydantic model for the structured Audit Context.
class AuditContextModel(BaseModel):
    audit_goal: str = Field(description="Refined, precise version of the user's goal.")
    image_description: Optional[str] = Field(None, description="Description of the reference image if provided, else null.")
    reference_is_composite_canvas: bool = Field(description="True if the reference image is a composite layout, webpage screenshot, hero banner, or real-world photograph containing the logo/asset. False if the reference image represents an isolated, standalone logo on a plain background.")
    inclusion_criteria: List[str] = Field(description="List of 3-7 specific, testable criteria an image MUST meet to be relevant.")
    exclusion_criteria: List[str] = Field(description="List of 2-5 criteria that EXCLUDE an image from relevance.")
    adjudication_logic: str = Field(description="A clear IF-THEN-ELSE statement defining PASS/FAIL conditions.")
    search_keywords: List[str] = Field(description="List of 5-10 single-word search terms (e.g. ['woman', 'female', 'portrait']) rather than multi-word phrases, to ensure broad keyword match capability in indexed assets.")
    vision_tag_filter: List[str] = Field(description="List of 2-5 keywords specifically mapped to Google Cloud Vision API tag vocabulary.")
    audit_instructions: str = Field(description="Detailed instructions to be passed to the auditing LLM describing the criteria.")
    extraction_schema: Dict[str, str] = Field(description="Dynamic key-value pairs representing additional boolean/integer/string properties to extract from the image to verify the audit criteria.")

def get_image_mime_type(path: str) -> str:
    """Helper to dynamically resolve visual asset MIME type based on file path extension."""
    lower_path = path.lower()
    if lower_path.endswith(".png"):
        return "image/png"
    elif lower_path.endswith(".webp"):
        return "image/webp"
    elif lower_path.endswith(".gif"):
        return "image/gif"
    return "image/jpeg"

def generate_audit_config(user_goal: str, reference_image_description: Optional[str] = None, available_tags: Optional[List[str]] = None) -> dict:
    """Translates a high-level user goal into a structured audit context with strict visual grounding and anti-hallucination guardrails."""
    image_context = ""
    if reference_image_description:
        image_context = f"\nReference Image Description: {reference_image_description}\n"

    active_tags = available_tags if available_tags else WEB_AUDIT_VISION_TAGS
    tags_pool_str = ", ".join([f"'{t}'" for t in active_tags])

    builder_prompt = f"""
You are a Lead Enterprise Visual & UI/UX Asset Auditor building a high-precision audit configuration for executive leadership.
Your task is to expand a user's audit goal into an airtight, zero-mistake structured audit context.

User Goal: {user_goal}
{image_context}

BRAND COMPLIANCE SIGNATURES INFERENCE:
Analyze the User Goal and the Reference Image Description (if provided) to extract:
1. The target brand, UI component, or visual subject under audit (e.g. Google Pay, YouTube, a specific Favicon, a cookies banner, or Google Workspace logos).
2. What constitutes the COMPLIANT (active, modern, approved) visual design, layout, typography, or shape.
3. What constitutes the NON-COMPLIANT (legacy, outdated, spoofed, or incorrect) visual design, layout, typography, or shape.
Ground all inclusion and exclusion criteria strictly in these inferred compliance signatures.

STEP 1: DYNAMIC BRAND & INTENT EXTRACTION
Identify the target brand/visual subject under audit based on the brand compliance signatures inference. Restrict all criteria, search keywords, and instructions strictly to this extracted subject.

STRICT VISUAL GROUNDING & ANTI-HALLUCINATION (CRITICAL):
- You MUST base all inclusion/exclusion criteria and visual descriptions strictly and exclusively on what is physically visible inside the provided reference image.
- Do NOT assume, extrapolate, or hallucinate the presence of brand names, wordmarks, UI elements, button texts, or logos that are cropped out or missing from the reference image.
- If a button, text, or logo is not visible in the reference image, do NOT include it as a mandatory requirement (inclusion criteria).

STEP 2: COMPLIANCE STATE ALIGNMENT & TEMPLATE ROLE ANALYSIS
Analyze the role of the reference image template (if provided) using your inferred compliance signatures.
- **State Conflict (Negation/Comparative Match)**: If the reference image represents the *Active/Compliant/New* standard, but the user wants to find *outdated/old* assets.
  * The template role is **Negative / Comparative Match**.
  * The exclusion criteria MUST exclude the reference image's compliant visual signatures.
  * The inclusion criteria MUST target older legacy styles of that same brand.
- **State Alignment (Positive Template Match)**: If the reference image represents the *Outdated/Legacy/Old* standard, and the user wants to find *outdated/old* assets.
  * The template role is **Positive Template Match**.
  * The inclusion criteria MUST require matching the reference image's legacy visual signatures.
  * The exclusion criteria MUST explicitly exclude the modern, compliant standard.
- **General Discovery Match**: If the user wants to find *all* assets of the brand regardless of state.
  * The inclusion criteria should pass the reference style AND other iterations of the brand.

STEP 3: REFERENCE COMPOSITE CANVAS DETECTION
Evaluate the description of the reference image:
- Set `reference_is_composite_canvas` to True if it describes a composite scene, real-world photograph, or webpage screenshot containing the logo/asset.
- Set `reference_is_composite_canvas` to False only if it represents an isolated, standalone logo on a plain background.

STEP 4: STRICT BRAND EXCLUSION & ANTI-SPOOFING GUARDRAIL (CRITICAL)
If the target visual subject is a specific sub-brand or product logo:
- You MUST explicitly include in the `exclusion_criteria` a rule to exclude the generic corporate master logo unless it is explicitly accompanied by the sub-brand.
- ANTI-SPOOFING: Add explicit rules to reject look-alike misspellings or related but incorrect sub-brands.

STEP 5: COLOR & CANVAS CONTEXT INDEPENDENCE (CRITICAL)
Unless the user's goal explicitly specifies a color constraint, you MUST explicitly write in the `audit_instructions` and `adjudication_logic` that color is not a match determinant.

CRITICAL RULE FOR vision_tag_filter:
You MUST ONLY select tags from the following list that exist in our database:
[{tags_pool_str}]

CRITICAL RULE FOR search_keywords (CRITICAL):
- The search_keywords MUST be a list of single-word search terms rather than multi-word phrases.

CRITICAL MANDATES FOR OPTICAL RESOLUTION & CLARITY AUDITS (CRITICAL):
1. **Strict Adjudication Logic**: Write a crystal-clear IF-THEN-ELSE statement in `adjudication_logic`.
   - Recognize Cropped/Blurred Targets: State explicitly that if the target visual asset is cropped, low-resolution, or blurred, but you can still identify its core structural signatures, it remains eligible.
   - Flag Resolution Status: In such cases, you must mark `optical_resolution_sufficient = False` in the extraction schema.
   - Only evaluate `matches_criteria = False` if the image is so extremely degraded, pixelated, or tiny (sub-pixel) that it is mathematically impossible to distinguish it from a generic shape.
2. **Diagnostic Extraction Schema**: In `extraction_schema`, define:
   - `detected_asset_style`: string (Exact description of what is seen on canvas)
   - `is_outdated_or_noncompliant`: boolean
   - `is_embedded_in_composite_hero`: boolean
   - `composite_location_notes`: string
   - `optical_resolution_sufficient`: boolean
"""

    # Passing the Pydantic model directly to the SDK
    response = client.models.generate_content(
        model=GEMINI_ORCHESTRATOR_MODEL,
        contents=builder_prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=AuditContextModel,
            temperature=0.0
        )
    )
    return json.loads(response.text)

def build_fused_query_text(context_dict: dict, is_negative: bool = False) -> str:
    """Combines all context fields into a single rich text representation for embedding."""
    if is_negative:
        return f"Exclude images that: {'; '.join(context_dict.get('exclusion_criteria', []))}. Specifically exclude spoofed misspellings, lookalike brands, and other products."

    parts = [
        f"Audit goal: {context_dict.get('audit_goal')}",
        f"Include images that: {'; '.join(context_dict.get('inclusion_criteria', []))}",
        "IMPORTANT FOR EMBEDDING SIMILARITY: Ignore foreground and background color differences. Focus purely on shape, text layout, logo structural design, and semantic meaning."
    ]
    if context_dict.get("image_description"):
        parts.append(f"Reference image: {context_dict.get('image_description')}")
    return " | ".join(parts)

async def embed_audit_context(context_dict: dict, reference_image_path: Optional[str] = None) -> Tuple[List[float], List[float]]:
    """Generates a POSITIVE and NEGATIVE multimodal embedding for Contrastive Retrieval."""
    pos_text = build_fused_query_text(context_dict, is_negative=False)
    neg_text = build_fused_query_text(context_dict, is_negative=True)

    pos_contents = []
    if reference_image_path:
        mime = get_image_mime_type(reference_image_path)
        if reference_image_path.startswith("gs://"):
            pos_contents.append(types.Part.from_uri(file_uri=reference_image_path, mime_type=mime))
        else:
            with open(reference_image_path, "rb") as f:
                pos_contents.append(types.Part.from_bytes(data=f.read(), mime_type=mime))

    pos_contents.append(pos_text[:1000])
    neg_contents = [neg_text[:1000]]

    def _embed(contents):
        return client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=contents,
            config=types.EmbedContentConfig(output_dimensionality=768)
        ).embeddings[0].values

    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=2) as executor:
        pos_future = loop.run_in_executor(executor, _embed, pos_contents)
        neg_future = loop.run_in_executor(executor, _embed, neg_contents)
        pos_vec, neg_vec = await asyncio.gather(pos_future, neg_future)

    return pos_vec, neg_vec


In [ ]:
# 2. Parallel Hybrid Search with RRF & Drop-Off Detection
from typing import List, Tuple, Optional
import numpy as np
import pandas as pd
from kneed import KneeLocator
from pgvector.asyncpg import register_vector
import asyncio
from concurrent.futures import ThreadPoolExecutor

async def run_semantic_reranking_and_filter(df_high: pd.DataFrame, df_edge: pd.DataFrame, audit_context: dict, max_workers: int = 25) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Applies LLM Cross-Encoder semantic scoring to the pre-segmented candidates, with visual vector safeguards to protect recall against description gaps."""
    candidates_df = pd.concat([df_high, df_edge], ignore_index=True)
    if candidates_df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    candidates = candidates_df.to_dict(orient="records")
    audit_goal = audit_context.get("audit_goal", "")

    def _score_candidate(row):
        # 1. Visual Vector Safeguard Check (Absolute visual distance check)
        # Cosine distance < 0.28 means high visual similarity (approx >0.72 similarity).
        # If this is triggered, we flag it to bypass Cross-Encoder checks.
        vec_dist = row.get("vector_distance", 1.0)
        if vec_dist < 0.28:
            return {
                **row,
                "cross_encoder_score": 100, # Max score to force-promote
                "relevance_score": row.get("relevance_score", 0.0) * 2.0, # Visual match boost
                "vector_safeguard_triggered": True
            }

        desc = row.get("gemini_description", "")
        tags = ", ".join(row.get("vision_tags") or [])
        filename = row.get("asset_filename", "")

        prompt = f"""You are a rapid relevance scoring engine.
Audit Goal: {audit_goal}
Candidate Description: {desc}
Candidate Tags: {tags}
Candidate Filename: {filename}

Score the candidate's textual relevance on a scale of 0 to 100 using this calibrated rubric:
- 80-100 (High): Direct matches to the target subject, clear presence of target visual elements, or standalone brand logos requested.
- 30-79 (Borderline): Contextual matches, related terms/brands, composite graphics containing the target, or candidate descriptions with some visual layout complexity.
- 0-29 (Low): Completely unrelated elements, different subjects (e.g. illustrations when looking for photos, different products, or unrelated graphics).

Output a single integer from 0 to 100 representing the score. Output NOTHING ELSE.
"""
        try:
            resp = client.models.generate_content(
                model=GEMINI_CROSS_ENCODER_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.0)
            )
            score = int(resp.text.strip())
        except:
            score = 50 # Default neutral fallback

        ce_mult = max(0.1, score / 50.0)
        new_score = row.get("relevance_score", 0.0) * ce_mult

        return {
            **row,
            "cross_encoder_score": score,
            "relevance_score": new_score,
            "vector_safeguard_triggered": False
        }

    print(f"[Cross-Encoder] Semantic reranking of {len(candidates)} candidate assets...")
    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        tasks = [loop.run_in_executor(executor, _score_candidate, item) for item in candidates]
        reranked = await asyncio.gather(*tasks)

    high_list = []
    edge_list = []
    low_list = []

    for item in reranked:
        if item.get("vector_safeguard_triggered", False):
            filename = item.get("asset_filename") or item["gcs_raw_path"].split("/")[-1]
            dist = item.get("vector_distance", 0)
            print(f"🛡️ Vector Safeguard triggered: Force-promoted {filename[:40]} due to high visual similarity (distance: {dist:.4f})")
            high_list.append(item)
            continue

        score = item["cross_encoder_score"]
        if score >= 75:
            high_list.append(item)
        elif score >= 30:
            edge_list.append(item)
        else:
            low_list.append(item)

    df_high_final = pd.DataFrame(high_list).sort_values(by="relevance_score", ascending=False).reset_index(drop=True) if high_list else pd.DataFrame()
    df_edge_final = pd.DataFrame(edge_list).sort_values(by="relevance_score", ascending=False).reset_index(drop=True) if edge_list else pd.DataFrame()
    df_low_final = pd.DataFrame(low_list).sort_values(by="relevance_score", ascending=False).reset_index(drop=True) if low_list else pd.DataFrame()

    return df_high_final, df_edge_final, df_low_final

def compute_weighted_rrf_rerank(candidates: list, audit_context: dict) -> list:
    """Zero-latency in-memory multi-factor reranker. Executes in local CPU RAM (< 0.5ms) without external API overhead."""
    tag_filter = [t.lower().strip() for t in audit_context.get("vision_tag_filter", []) if t]
    search_keywords = [k.lower().strip() for k in audit_context.get("search_keywords", []) if k]
    reranked = []

    for item in candidates:
        row = item["data"]
        base_score = item["score"]
        multiplier = 1.0

        # 1. Vision tag exact hit boost (2.0x per matching tag up to 8x)
        tags = [str(t).lower() for t in (row.get("vision_tags") or [])]
        tag_hits = sum(1 for t in tags if any(ft in t or t in ft for ft in tag_filter))
        if tag_hits > 0:
            multiplier *= (2.0 ** min(tag_hits, 3))

        # 2. Keyword exact hit in filename or description boost (1.5x per matching keyword up to 2.25x)
        desc = str(row.get("gemini_description") or "").lower()
        fname = str(row.get("asset_filename") or "").lower()
        kw_hits = sum(1 for kw in search_keywords if kw in desc or kw in fname)
        if kw_hits > 0:
            multiplier *= min(1.5 ** kw_hits, 2.25)

        reranked.append({
            "data": row,
            "score": base_score * multiplier,
            "base_rrf_score": base_score,
            "rerank_multiplier": multiplier,
            "vector_distance": item.get("vector_distance", 1.0)
        })

    reranked.sort(key=lambda x: x["score"], reverse=True)
    return reranked

async def run_hybrid_search(scope_config: dict, audit_context: dict, reference_image_path: Optional[str] = None, limit: int = 10000) -> List[dict]:
    """Performs parallel 3-Arm Vector + Keyword + Tag Boosting search with RRF fusion, local reranking, and deduplication (No silent cliff)."""
    pos_vec, neg_vec = await embed_audit_context(audit_context, reference_image_path)

    engine, _ = await get_alloydb_connection()

    search_keywords = audit_context.get("search_keywords", [])
    keyword_query_str = " OR ".join(search_keywords) if search_keywords else ""

    tag_filter = audit_context.get("vision_tag_filter", [])
    tag_filter = tag_filter if tag_filter else []

    # Run the 3 database query arms concurrently using pooled SQLAlchemy connections.
    async def run_vector():
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            await register_vector(db)
            rows = await db.fetch(
                f"""
                SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash,
                       (embedding <=> $1::vector) as vector_distance
                FROM {DB_SCHEMA}.visual_assets
                ORDER BY $1::vector <=> embedding - (0.3 * (embedding <=> $2::vector)) ASC
                LIMIT $3
                """,
                pos_vec, neg_vec, limit
            )
            return [dict(r) for r in rows]

    async def run_fts():
        if not keyword_query_str:
            return []
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            rows = await db.fetch(
                f"""
                SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash
                FROM {DB_SCHEMA}.visual_assets
                WHERE to_tsvector('english', gemini_description) @@ websearch_to_tsquery('english', $1)
                LIMIT $2
                """,
                keyword_query_str, limit
            )
            return [dict(r) for r in rows]

    async def run_tags():
        if not tag_filter:
            return []
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            rows = await db.fetch(
                f"""
                SELECT asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash
                FROM {DB_SCHEMA}.visual_assets
                WHERE EXISTS (
                    SELECT 1 FROM unnest(vision_tags) tag
                    WHERE EXISTS (
                        SELECT 1 FROM unnest($1::text[]) filter_tag
                        WHERE tag ILIKE '%' || filter_tag || '%'
                    )
                )
                LIMIT $2
                """,
                tag_filter, limit
            )
            return [dict(r) for r in rows]

    vector_task = run_vector()
    fts_task = run_fts()
    tag_task = run_tags()

    vector_results, fts_results, tag_results = await asyncio.gather(vector_task, fts_task, tag_task)

    promoted_ids = set()
    for r in fts_results[:100]:
        promoted_ids.add(str(r["asset_id"]))
    for r in tag_results[:100]:
        promoted_ids.add(str(r["asset_id"]))

    # 3-Arm RRF Fusion (k=60)
    k = 60
    results_map = {}

    # Pre-populate with vector distances
    vector_distance_map = {str(r["asset_id"]): r["vector_distance"] for r in vector_results}

    def upsert_ranks(results_list, weight=1.0):
        for rank, row in enumerate(results_list):
            img_id = str(row["asset_id"])
            if img_id not in results_map:
                results_map[img_id] = {"data": row, "score": 0.0, "vector_distance": vector_distance_map.get(img_id, 1.0)}
            results_map[img_id]["score"] += weight / (k + rank + 1)

    upsert_ranks(vector_results, weight=0.60)
    if fts_results:
        upsert_ranks(fts_results, weight=0.25)
    if tag_results:
        upsert_ranks(tag_results, weight=0.15)

    fused = list(results_map.values())

    # Zero-Latency In-Memory Reranking (Provides a smooth score curve for Kneedle)
    reranked_fused = compute_weighted_rrf_rerank(fused, audit_context)

    # Deduplication
    seen_identifiers = set()
    deduplicated = []
    for item in reranked_fused:
        row = item["data"]
        img_id = str(row["asset_id"])
        img_identifier = row.get("content_hash") or row.get("gcs_raw_path")
        if img_identifier not in seen_identifiers:
            seen_identifiers.add(img_identifier)
            is_promoted = img_id in promoted_ids
            deduplicated.append({
                **row,
                "relevance_score": item["score"],
                "base_rrf_score": item.get("base_rrf_score", 0),
                "rerank_multiplier": item.get("rerank_multiplier", 1),
                "promoted_by_keyword_or_tag": is_promoted,
                "vector_distance": item.get("vector_distance", 1.0)
            })

    return deduplicated[:limit]

def detect_dropoff_flawless(df: pd.DataFrame, sensitivity: float = 1.0) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Applies Kneedle curvature + rolling volatility to segment candidates into High, Edge, and Low."""
    if df is None or df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    df_sorted = df.sort_values(by="relevance_score", ascending=False).reset_index(drop=True)
    y = df_sorted["relevance_score"].values
    x = np.arange(len(y))

    y_min, y_max = y.min(), y.max()
    if y_max == y_min:
        return df_sorted.iloc[:int(len(y)*0.3)], df_sorted.iloc[int(len(y)*0.3):int(len(y)*0.6)], df_sorted.iloc[int(len(y)*0.6):]

    y_norm = (y - y_min) / (y_max - y_min + 1e-9)
    x_norm = x / (len(x) - 1)

    coords = np.column_stack((x_norm, y_norm))
    line_start, line_end = coords[0], coords[-1]
    line_vec = line_end - line_start
    line_vec_norm = line_vec / np.sqrt(np.sum(line_vec**2))
    vec_from_start = coords - line_start
    scalar_proj = np.dot(vec_from_start, line_vec_norm)
    proj_on_line = np.outer(scalar_proj, line_vec_norm)
    dist_to_line = np.sqrt(np.sum((coords - proj_on_line)**2, axis=1))

    idx1 = np.argmax(dist_to_line)

    window = max(3, int(len(y) * 0.05))
    rolling_std = pd.Series(y_norm).rolling(window=window, center=True).std().fillna(0).values
    noise_threshold = np.mean(rolling_std) * (0.6 / sensitivity)

    idx2 = len(y) - 1
    for i in range(idx1 + 2, len(rolling_std)):
        if rolling_std[i] < noise_threshold:
            idx2 = i
            break

    min_borderline_width = max(10, int((len(y) - idx1) * 0.25))
    if (idx2 - idx1) < min_borderline_width:
        idx2 = min(len(y) - 1, idx1 + min_borderline_width)

    idx1 = max(10, idx1)

    high_df = df_sorted.iloc[:idx1 + 1].copy()
    edge_df = df_sorted.iloc[idx1 + 1: idx2 + 1].copy()
    low_df = df_sorted.iloc[idx2 + 1:].copy()

    # Visual Vector Safeguard: Check if any candidate has extremely high similarity (distance < 0.28)
    # even if it is currently classified in Low_df (or has been discarded).
    # Force rescue these to protect visual recall.
    if "vector_distance" in low_df.columns:
        rescued_vec = low_df[low_df["vector_distance"] < 0.28].copy()
        if not rescued_vec.empty:
            edge_df = pd.concat([edge_df, rescued_vec], ignore_index=True)
            low_df = low_df[low_df["vector_distance"] >= 0.28].copy()
            print(f"🛡️ Vector Safeguard triggered in Kneedle: Force-rescued {len(rescued_vec)} candidate(s) from Low to Borderline based on high visual similarity.")

    if "promoted_by_keyword_or_tag" in low_df.columns:
        rescued = low_df[low_df["promoted_by_keyword_or_tag"] == True].copy()
        if not rescued.empty:
            edge_df = pd.concat([edge_df, rescued], ignore_index=True)
            low_df = low_df[low_df["promoted_by_keyword_or_tag"] != True].copy()
            print(f"🛡️ Safeguard triggered: Promoted {len(rescued)} composite/diluted candidates from Low to Borderline tier based on exact keyword/tag match.")

    return high_df, edge_df, low_df


## Retrieval, Reranking, and Drop-off Segmentation Engine

This cell defines the core search engine:

- `run_hybrid_search`: Combines Vector, Full-Text, and Tag search arms into a fused RRF list, applying zero-latency keyword boosts.
- `detect_dropoff_flawless`: Curvature-based Kneedle algorithm that truncates irrelevant tail results.

In [ ]:
# 3. Hydration & Parallel LLM Audit Inference
import asyncio
from concurrent.futures import ThreadPoolExecutor
import json
import pandas as pd
from typing import List, Tuple, Optional
from google.genai import types
from google.cloud import storage
from pydantic import create_model, Field
import time
from PIL import Image
import io

def process_transparency(image_bytes: bytes, default_bg: Tuple[int, int, int] = (30, 30, 30)) -> bytes:
    """Detects alpha transparency in an image and composites it onto a solid dark background to ensure light elements/text remain visible."""
    try:
        img = Image.open(io.BytesIO(image_bytes))
        if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
            img = img.convert('RGBA')
            # Create a solid dark grey background
            bg = Image.new("RGBA", img.size, default_bg + (255,))
            alpha_composite = Image.alpha_composite(bg, img)
            final_img = alpha_composite.convert("RGB")
            out_bytes = io.BytesIO()
            final_img.save(out_bytes, format="PNG")
            return out_bytes.getvalue()
    except Exception as e:
        print(f"Warning: Failed to preprocess image transparency: {e}")
    return image_bytes

def enforce_zero_false_positives_rules(df: pd.DataFrame) -> pd.DataFrame:
    """Enforces zero-false-positive criteria. Demotes matches if confidence is below 75%."""
    if df.empty:
        return df

    def guardrail_check(row):
        matches = bool(row.get("matches_criteria", False))
        conf = int(row.get("match_confidence", 100))
        rationale = str(row.get("visual_analysis_step_by_step", "")) + " " + str(row.get("match_rationale", ""))
        if matches and conf < 75:
            row["matches_criteria"] = False
            row["match_rationale"] = f"[GUARDRAIL DEMOTION: Conf {conf}% < 75%] {rationale}"
        return row

    return df.apply(guardrail_check, axis=1)

async def run_llm_audit_single(asset_data: dict, audit_config: dict, _executor=None, reference_image_part: Optional[types.Part] = None) -> dict:
    """Evaluates a single image asset against dynamic JSON schema using Pydantic, supporting side-by-side reference comparisons and transparency blending."""
    audit_instructions = audit_config.get("audit_instructions", "")
    extraction_schema = audit_config.get("extraction_schema", {})
    inclusion_criteria = audit_config.get("inclusion_criteria", [])
    exclusion_criteria = audit_config.get("exclusion_criteria", [])
    adjudication_logic = audit_config.get("adjudication_logic", "")
    is_composite = bool(audit_config.get("reference_is_composite_canvas", False))

    inclusion_str = "\n".join([f"- {c}" for c in inclusion_criteria]) if inclusion_criteria else "- None specified"
    exclusion_str = "\n".join([f"- {c}" for c in exclusion_criteria]) if exclusion_criteria else "- None specified"

    fields = {
        "visual_analysis_step_by_step": (
            str,
            Field(description="CHAIN OF THOUGHT: Write a comprehensive, step-by-step visual analysis of the image BEFORE making any conclusions. Describe exact shapes, typography, brand marks, colors, and layout.")
        ),
        "matches_criteria": (
            bool,
            Field(description="Strict final evaluation: True ONLY if the asset matches the target criteria and passes adjudication_logic (even inside a composite hero banner). False otherwise.")
        ),
        "match_confidence": (
            int,
            Field(description="Match Confidence percentage (0-100%). You must assign < 75 if there is any doubt or visual occlusion.")
        ),
        "match_rationale": (
            str,
            Field(description="A concise final executive rationale explaining exactly why matches_criteria evaluated to True or False based on the visual_analysis_step_by_step.")
        )
    }

    for field_name, field_info in extraction_schema.items():
        if field_name in ["matches_criteria", "match_confidence", "match_rationale", "visual_analysis_step_by_step"]:
            continue

        t = str
        desc = f"Extracted value for {field_name}"

        if isinstance(field_info, dict):
            ftype = field_info.get("field_type", "string").lower()
            desc = field_info.get("description", desc)
        else:
            ftype = str(field_info).lower()

        if ftype == "boolean":
            t = bool
        elif ftype == "integer":
            t = int
        elif ftype == "number":
            t = float

        fields[field_name] = (t, Field(description=desc))

    DynamicAuditModel = create_model("DynamicAuditModel", **fields)

    reference_instructions = ""
    if reference_image_part:
        if is_composite:
            reference_instructions = f"""
You are given two images:
- The FIRST image (image_0) is the Reference Image (the template provided by the user).
- The SECOND image (image_1) is the Candidate Image (the asset under audit).

REFERENCE TEMPLATE DETAILS:
- Audit Goal: {audit_config.get('audit_goal')}
- Reference Target Description: {audit_config.get('image_description', 'No description')}

CANVAS ISOLATION DIRECTIVE (CRITICAL):
The reference image (image_0) is a **Composite Canvas** (a complex real-world photograph/screenshot containing the logo).
Do NOT expect the candidate image under audit (image_1) to contain the hands, terminals, backgrounds, or full layout seen in image_0.
Instead, look at the target logo/brand style (e.g. logos or wordmark shown on the phone screen) inside image_0.
Verify if the candidate image (image_1) contains that target logo style. Ignore all other background visual noise in image_0.
"""
        else:
            reference_instructions = f"""
You are given two images:
- The FIRST image (image_0) is the Reference Image (the template provided by the user).
- The SECOND image (image_1) is the Candidate Image (the asset under audit).

REFERENCE TEMPLATE DETAILS:
- Audit Goal: {audit_config.get('audit_goal')}
- Reference Image Description: {audit_config.get('image_description', 'No description')}

CANVAS ISOLATION DIRECTIVE:
Since the Reference Image (image_0) is a standalone logo, compare the candidate image (image_1) side-by-side against image_0.
"""

    # Updated prompt template: Objective, clean, and safe from prompt override classifications
    audit_prompt = f"""
Evaluate this image against the specified audit goal and criteria with high precision.

{reference_instructions}

[Color Independence Rule]:
Unless the Inclusion Criteria explicitly mention a required color, you must ignore any color differences between the reference image and the candidate image.

[Exact Typography Rule]:
Pay strict attention to typography and spelling (e.g. 'Google Play' is NOT 'Google Pay', 'Ads' is NOT 'AdWords'). Reject any assets that contain lookalike or misspelled branding unless the inclusion criteria explicitly permit them.

AUDIT SCOPE & EVALUATION RULES:

[Inclusion Criteria – Asset MUST fulfill these to pass]:
{inclusion_str}

[Exclusion Criteria – If asset triggers any of these, it MUST fail]:
{exclusion_str}

[Strict Adjudication Rule]:
{adjudication_logic}

[General Audit Instructions]:
{audit_instructions}

EXECUTION STEPS:
1. Provide a detailed, step-by-step visual analysis of the image in the `visual_analysis_step_by_step` field. Scan the entire canvas to inspect all visual details, shapes, and text.
2. Extract all diagnostic visual features requested in the schema, including whether the target is embedded inside a composite hero graphic (`is_embedded_in_composite_hero`).
3. Apply the Strict Adjudication Rule against your extracted visual findings.
4. Set `matches_criteria` to true if the asset satisfies the inclusion criteria without triggering any exclusion criteria with 100% certainty.
5. Provide a clear justification in `match_rationale` explaining your decision based on your step-by-step analysis.
"""

    gcs_path = asset_data["gcs_raw_path"]

    # Resolve bytes locally or from GCS to handle transparency rendering
    def _download_and_preprocess():
        if gcs_path.startswith("gs://"):
            bucket_name = gcs_path.split("/")[2]
            blob_name = "/".join(gcs_path.split("/")[3:])
            client_storage = storage.Client(project=PROJECT_ID)
            bucket = client_storage.bucket(bucket_name)
            blob = bucket.blob(blob_name)
            img_bytes = blob.download_as_bytes()
        else:
            with open(gcs_path, "rb") as f:
                img_bytes = f.read()
        return process_transparency(img_bytes)

    loop = asyncio.get_running_loop()
    try:
        # Download and composite image on background thread
        processed_bytes = await loop.run_in_executor(_executor, _download_and_preprocess)
        image_part = types.Part.from_bytes(data=processed_bytes, mime_type="image/png")

        contents = []
        if reference_image_part:
            contents.append(reference_image_part)

        contents.append(image_part)
        contents.append(audit_prompt)

        def _call_gemini():
            return client.models.generate_content(
                model=GEMINI_INFERENCE_MODEL,
                contents=contents,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=DynamicAuditModel,
                    temperature=0.0
                )
            )

        response = await loop.run_in_executor(_executor, _call_gemini)
        extracted_data = json.loads(response.text)
    except Exception as e:
        extracted_data = {
            "matches_criteria": False,
            "match_confidence": 0,
            "match_rationale": f"Audit evaluation failed due to error: {str(e)}",
            "error": str(e)
        }

    return {**asset_data, **extracted_data}

async def run_llm_inference_on_dropoff_results(df_high: pd.DataFrame, df_edge: pd.DataFrame, audit_config: dict, max_workers: int = 15, reference_image_path: Optional[str] = None) -> pd.DataFrame:
    """Runs parallel multi-threaded LLM inference on candidate subsets, supporting side-by-side template matches, GCS/local transparency preprocessing, and caching."""
    candidates_df = pd.concat([df_high, df_edge], ignore_index=True)
    if candidates_df.empty:
        return pd.DataFrame()

    candidates = candidates_df.to_dict(orient="records")

    print(f"Starting parallel LLM audit inference on {len(candidates)} candidates (Max concurrency: {max_workers})...")

    # Helper to download and preprocess the reference image once
    def _get_reference_part():
        if reference_image_path.startswith("gs://"):
            bucket_name = reference_image_path.split("/")[2]
            blob_name = "/".join(reference_image_path.split("/")[3:])
            client_storage = storage.Client(project=PROJECT_ID)
            bucket = client_storage.bucket(bucket_name)
            blob = bucket.blob(blob_name)
            ref_bytes = blob.download_as_bytes()
        else:
            with open(reference_image_path, "rb") as f:
                ref_bytes = f.read()
        processed_ref = process_transparency(ref_bytes)
        return types.Part.from_bytes(data=processed_ref, mime_type="image/png")

    t0 = time.time()
    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        reference_image_part = None
        if reference_image_path:
            reference_image_part = await loop.run_in_executor(executor, _get_reference_part)

        tasks = [run_llm_audit_single(asset, audit_config, _executor=executor, reference_image_part=reference_image_part) for asset in candidates]
        results = await asyncio.gather(*tasks)

    duration = time.time() - t0
    print(f" Processing assets... [{len(candidates)}/{len(candidates)}] completed")
    print(f" Inference completed. Enforcing Zero-FP policies and sorting scoreboard...")

    results_df = pd.DataFrame(results)
    results_df = enforce_zero_false_positives_rules(results_df)

    results_df = results_df.sort_values(by="relevance_score", ascending=False).reset_index(drop=True)

    print(f"\n=== VERIFIED AUDIT SCOREBOARD (Sorted by Search Similarity) ===")
    for i, r in results_df.iterrows():
        status_label = "PASS" if r.get("matches_criteria", False) else "FAIL"
        filename = r.get("asset_filename") or r.get("gcs_raw_path", "").split("/")[-1]
        sim = r.get("relevance_score", 0.0)
        conf = r.get("match_confidence", 100)
        print(f"[{i+1}/{len(results_df)}] {status_label} | {filename[:40]} | Sim: {sim:.4f} | Conf: {conf}%")

    return results_df


In [ ]:
# 4. Calibration Summary & Audit Results Saving E2E Loop
async def save_audit_results_to_db(session_id: str, results_df: pd.DataFrame):
    """Saves the final audited results into AlloyDB (Bypassed by default in the interactive runner)."""
    if results_df.empty:
        return

    engine, _ = await get_alloydb_connection()
    async with engine.connect() as conn:
        raw_conn = await conn.get_raw_connection()
        db = raw_conn.driver_connection
        records = results_df.to_dict("records")
        for r in records:
            verdict = "PASS" if r.get("matches_criteria") is True else "FAIL"

            # Extract criteria_checks from JSON response or construct it
            criteria_checks = {k: v for k, v in r.items() if k not in ["asset_id", "gcs_raw_path", "matches_criteria", "error", "asset_filename", "page_url"]}

            await db.execute(
                f"""
                INSERT INTO {DB_SCHEMA}.audit_results (
                    session_id, asset_id, overall_verdict, adjudication_result,
                    criteria_checks, rationale, confidence_band
                ) VALUES ($1, $2, $3, $4, $5, $6, $7)
                """,
                session_id,
                r.get("asset_id"),
                verdict,
                r.get("matches_criteria", False),
                json.dumps(criteria_checks),
                r.get("match_rationale", "Completed"),
                "high" if r.get("match_confidence", 0) > 95 else ("borderline" if r.get("match_confidence", 0) >= 70 else "below_threshold")
            )
    print("Audit results saved to database.")

async def generate_ai_audit_summary(results_df: pd.DataFrame, audit_config: dict) -> str:
    """Generates an AI-powered executive summary of the visual asset audit results."""
    import json
    import numpy as np
    if results_df is None or results_df.empty:
        return "No audit results available to generate a summary."

    # Select columns to pass to the LLM, avoiding internal or verbose vector columns
    exclude_cols = {"num_chunks", "max_relevance_score", "gcs_raw_path", "gcs_processed_path", "embedding", "embedding_at"}
    cols_to_include = [col for col in results_df.columns if col not in exclude_cols]

    # Convert to records safely, handling NumPy arrays, lists, and floats without boolean truth value ambiguity
    clean_df = results_df[cols_to_include].copy()
    for col in clean_df.columns:
        def safe_clean(val):
            if val is None:
                return None
            if isinstance(val, (np.ndarray, pd.Series)):
                return val.tolist() if val.size > 0 else None
            if isinstance(val, float) and pd.isna(val):
                return None
            return val
        clean_df[col] = clean_df[col].apply(safe_clean)

    records = clean_df.to_dict(orient="records")
    formatted_results = json.dumps(records, indent=2)

    audit_instructions = audit_config.get("audit_instructions", "No specific audit context provided.")

    # Compute basic stats to seed in the prompt
    total_audited = len(results_df)
    matches_col = "matches_criteria" if "matches_criteria" in results_df.columns else None
    if matches_col:
        # Convert to boolean safely, handling string representation if any
        matches_true = results_df[matches_col].apply(lambda x: str(x).lower() in ("true", "1", "yes")).sum()
    else:
        matches_true = "N/A"

    summary_prompt = f"""
You are a Lead Visual Asset Auditor.
Your task is to write a visually engaging, highly structured, and extremely concise summary of a visual asset audit Test Bench calibration run.

STRICT RULES FOR FORMATTING & SECTIONS:
1. You MUST only include the following exact three sections in the output:
   - Objective & Scope (Preview Subset) (within the top [!NOTE] block)
   - Calibration Statistics (as a numbered list)
   - Configuration Calibration Insights (as a single, brief narrative paragraph of 3-4 sentences detailing the visual rules performance)
2. DO NOT include any other sections.
3. DO NOT pass any definitive verdicts of success.

AUDIT CONTEXT (Visual Evaluation Criteria):
{audit_instructions}

TEST BENCH STATS:
- Total images audited: {total_audited}
- Total images matching criteria (True): {matches_true}

TEST BENCH FINDINGS (JSON):
{formatted_results}
"""

    response = client.models.generate_content(
        model=GEMINI_ORCHESTRATOR_MODEL,
        contents=summary_prompt,
        config=types.GenerateContentConfig(temperature=0.0)
    )
    return response.text


## Telemetry Summaries & Calibration Database Saving

This cell defines functions to generate final executive AI summaries (`generate_ai_audit_summary`) and save audit session details into the run calibration tables in AlloyDB for telemetry history.

In [ ]:
# 5. E2E Execution Helpers (Interactive Split Stages)
import base64
import os
import time
import json
import pandas as pd
from google.cloud import storage

def get_gcs_image_base64(gcs_path: str) -> str:
    """Downloads image from GCS or local file and returns its base64 data URI for inline HTML rendering."""
    try:
        image_bytes = b""
        mime_type = "image/png"

        # Detect format
        lower_path = gcs_path.lower()
        if lower_path.endswith(".jpg") or lower_path.endswith(".jpeg"):
            mime_type = "image/jpeg"
        elif lower_path.endswith(".webp"):
            mime_type = "image/webp"
        elif lower_path.endswith(".gif"):
            mime_type = "image/gif"

        if gcs_path.startswith("gs://"):
            parts = gcs_path.replace("gs://", "").split("/", 1)
            bucket_name = parts[0]
            blob_name = parts[1]

            # Authenticated download using python client
            storage_client = storage.Client()
            bucket = storage_client.bucket(bucket_name)
            blob = bucket.blob(blob_name)
            image_bytes = blob.download_as_bytes()
        elif os.path.exists(gcs_path):
            with open(gcs_path, "rb") as f:
                image_bytes = f.read()
        else:
            return ""

        encoded = base64.b64encode(image_bytes).decode("utf-8")
        return f"data:{mime_type};base64,{encoded}"
    except Exception as e:
        return ""

async def generate_audit_config_only(user_goal: str, reference_image_path: Optional[str] = None) -> dict:
    """Stage 1: Analyzes reference image (Forensic) and generates structured Audit Configuration."""
    reference_image_description = None
    if reference_image_path:
        print("0. Performing forensic executive analysis on reference image...")
        ref_mime = get_image_mime_type(reference_image_path)
        if reference_image_path.startswith("gs://"):
            image_part = types.Part.from_uri(file_uri=reference_image_path, mime_type=ref_mime)
        else:
            with open(reference_image_path, "rb") as f:
                image_part = types.Part.from_bytes(data=f.read(), mime_type=ref_mime)

        forensic_analysis_prompt = """You are a Lead Executive Visual & UI/UX Inspector. Perform an exhaustive, forensic-level breakdown of this uploaded reference image for enterprise audit configuration.
Provide a comprehensive, structured analysis starting with Brand Hierarchy:
1. Target Brand & Scope: What exact brand or product is shown? (e.g. specifically Google Pay / G Pay, or Google Workspace). Do not mix in separate products.
2. Primary Asset Hierarchy & Category: Classify whether this image is a Standalone Brand Logo, a UI Component (Payment Button, Cookie Modal), or a Composite Canvas (Hero Banner, Screenshot, Collage).
3. Exact Visual Signatures: Detail exact wordmarks, typography, overlapping geometric shapes (e.g. interlocking loops vs text wordmark), border radius, padding, and layout structure.
4. Color & Contrast Styling: Detail exact colors, contrast levels, and shadow/elevation effects.
5. Compliance & Style Classification: Classify whether this image represents an active/compliant style or an outdated/deprecated legacy pattern.
-Guideline for Google Pay (G Pay) compliance status:*
- The CURRENT COMPLIANT standard is the multi-colored interlocking loops design (four colored curved segments in blue, red, yellow, green forming a stylized double-loop G/Pay shape).
- Any logo showing the 'Google Pay' or 'G Pay' wordmark (where 'Google' is multi-colored and 'Pay' is grey/black/white) is an OUTDATED/LEGACY pattern."""

        resp = client.models.generate_content(
            model=GEMINI_ORCHESTRATOR_MODEL,
            contents=[image_part, forensic_analysis_prompt],
            config=types.GenerateContentConfig(temperature=0.0)
        )
        reference_image_description = resp.text
        print(f"Forensic Reference Image Analysis:\n{reference_image_description}\n{'='*50}")

    # Dynamically query all unique tags from AlloyDB visual_assets
    available_tags = []
    try:
        engine, _ = await get_alloydb_connection()
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            rows = await db.fetch(
                f"SELECT DISTINCT tag FROM {DB_SCHEMA}.visual_assets, unnest(vision_tags) tag"
            )
            available_tags = [r["tag"] for r in rows if r["tag"]]
            print(f"Loaded {len(available_tags)} unique tag vocabulary terms from database.")
    except Exception as e:
        print(f"Warning: Could not fetch unique tags from DB ({e}). Using static fallback.")

    print("1. Translating goal into Audit Configuration...")
    audit_config = generate_audit_config(user_goal, reference_image_description, available_tags)
    print("Generated Configuration:\n", json.dumps(audit_config, indent=2))
    return audit_config

async def run_full_test_bench_pipeline_execution(audit_config: dict, reference_image_path: Optional[str] = None, quick_mode: bool = False) -> pd.DataFrame:
    """Stage 2, 3, 3.5, 4, 5: Executes Retrieval, Semantic Reranking, Inference, and Calibration scorecards."""
    pipeline_start_time = time.time()
    telemetry = {}

    t0 = time.time()
    print("\n2. Running Parallel 3-Arm Hybrid Search with RRF...")
    search_results = await run_hybrid_search({}, audit_config, reference_image_path)
    telemetry["Stage 2 (3-Arm RRF Retrieval)"] = f"{time.time() - t0:.2f}s | {len(search_results)} candidates retrieved"
    print(f"Found {len(search_results)} candidates.")

    t0 = time.time()
    print("\n3. Running Drop-off Analysis (Kneedle + Volatility)...")
    df_results = pd.DataFrame(search_results)
    df_high, df_edge, df_low = detect_dropoff_flawless(df_results)
    telemetry["Stage 3 (Drop-off Segmentation)"] = f"{time.time() - t0:.2f}s | High: {len(df_high)}, Borderline: {len(df_edge)}, Low (Filtered): {len(df_low)}"
    print(f"Rough Candidates -> High: {len(df_high)} | Borderline: {len(df_edge)} | Low: {len(df_low)}")

    t0 = time.time()
    print("\n3.5. Running Semantic Reranking and Fine Filtering...")
    df_high_sem, df_edge_sem, df_low_sem = await run_semantic_reranking_and_filter(df_high, df_edge, audit_config)
    telemetry["Stage 3.5 (Semantic Reranking)"] = f"{time.time() - t0:.2f}s | High: {len(df_high_sem)}, Borderline: {len(df_edge_sem)}"
    print(f"Semantic Candidates -> High: {len(df_high_sem)} | Borderline: {len(df_edge_sem)} | Low (Discarded): {len(df_low_sem)}, Low (Demoted): {len(df_low_sem)}")

    # Save full results globally for evaluation recall calculation
    globals()["df_results_full"] = df_results

    # Apply Quick Mode vs Smart Scan selection
    if quick_mode:
        print("\n Quick Mode enabled: Selecting top 30 High confidence and top 30 Borderline for visual inference.")
        candidates_high = df_high_sem.head(30) if not df_high_sem.empty else pd.DataFrame()
        candidates_edge = df_edge_sem.head(30) if not df_edge_sem.empty else pd.DataFrame()
    else:
        print("\n Smart Scan enabled: Selecting all High confidence and all Borderline for full visual inference.")
        candidates_high = df_high_sem
        candidates_edge = df_edge_sem

    t0 = time.time()
    print("\n4. Running Parallel LLM Audit Inference...")
    results_df = await run_llm_inference_on_dropoff_results(candidates_high, candidates_edge, audit_config, reference_image_path=reference_image_path)
    inf_duration = time.time() - t0
    throughput = len(results_df) / inf_duration if inf_duration > 0 else 0
    telemetry["Stage 4 (Parallel Visual Inference)"] = f"{inf_duration:.2f}s | Throughput: {throughput:.2f} images/sec"
    print(f"LLM Results: {len(results_df)} assets audited.")

    t0 = time.time()
    print("\n5. Generating Calibration Summary...")
    summary = await generate_ai_audit_summary(results_df, audit_config)
    telemetry["Stage 5 (Executive Summary Generation)"] = f"{time.time() - t0:.2f}s"

    total_time = time.time() - pipeline_start_time

    # Compute Observability Stats
    error_count = results_df["error"].notna().sum() if (not results_df.empty and "error" in results_df.columns) else 0
    high_conf = (results_df["match_confidence"] >= 95).sum() if (not results_df.empty and "match_confidence" in results_df.columns) else 0
    borderline_conf = ((results_df["match_confidence"] >= 70) & (results_df["match_confidence"] < 95)).sum() if (not results_df.empty and "match_confidence" in results_df.columns) else 0
    low_conf = (results_df["match_confidence"] < 70).sum() if (not results_df.empty and "match_confidence" in results_df.columns) else 0

    try:
        from IPython.display import display, Markdown, HTML

        telemetry_md = f"""
### 📊 Enterprise Observability & Telemetry Scorecard
| Stage / Metric | Value / Duration | Status |
|:--- |:--- |:---|
| **Total Pipeline Execution Time** | **{total_time:.2f}s** | 🟢 Optimal Throughput |
| **Parallel Inference Speed** | **{throughput:.2f} images/sec** | Concurrency: 15 Workers |
| **Stage 2: 3-Arm RRF Retrieval** | {telemetry.get('Stage 2 (3-Arm RRF Retrieval)', 'N/A')} | HNSW + FTS + Tag Boost |
| **Stage 3: Kneedle Segmentation** | {telemetry.get('Stage 3 (Drop-off Segmentation)', 'N/A')} | Noise Tail Truncated |
| **Stage 3.5: Semantic Reranking** | {telemetry.get('Stage 3.5 (Semantic Reranking)', 'N/A')} | LLM Text Cross-Encoder |
| **Stage 4: Vision Inference** | {telemetry.get('Stage 4 (Parallel Visual Inference)', 'N/A')} | 0 False Positive Verdict Guarantee |
| **Confidence Band Distribution** | High (>95%): **{high_conf}** || Borderline (70-95%): **{borderline_conf}** || Low (<70%): **{low_conf}** || Error Count: **{error_count}** |
"""
        display(Markdown(telemetry_md))
        display(Markdown(summary))

        if not results_df.empty:
            display(Markdown("### Detailed Audit Results"))
            display_df = results_df.copy()
            display_df["Visual Preview"] = display_df["gcs_raw_path"].apply(
                lambda x: f'<img src="{get_gcs_image_base64(x)}" width="150" />' if get_gcs_image_base64(x) else '[No Preview]'
            )
            display_df["Page Link"] = display_df["page_url"].apply(
                lambda x: f'<a href="{x}" target="_blank">{x}</a>' if x else '[No Page Link]'
            )
            cols = ["Visual Preview", "matches_criteria", "relevance_score", "gcs_raw_path", "Page Link", "match_rationale"]
            col_labels = ["Visual Preview", "Matches Criteria", "Search Similarity", "GCS Path", "Page Location URL", "Rationale"]

            if "gemini_description" in display_df.columns:
                cols.append("gemini_description")
                col_labels.append("Image Description")

            display_df = display_df[cols]
            display_df.columns = col_labels

            display(HTML(display_df.to_html(escape=False, index=False)))
    except (ImportError, ModuleNotFoundError):
        print("\n=== TELEMETRY SCORECARD ===")
        for k, v in telemetry.items():
            print(f"  {k}: {v}")
        print(f"  Total Time: {total_time:.2f}s | Errors: {error_count}")
        print("\n=== CALIBRATION SUMMARY ===")
        print(summary)

    return results_df

# Bypassed original monolithic E2E pipeline name to map to partitioned runners
async def run_full_test_bench_pipeline(user_goal: str, reference_image_path: Optional[str] = None, quick_mode: bool = False) -> pd.DataFrame:
    """Wrapper to maintain backwards compatibility for existing cells."""
    config = await generate_audit_config_only(user_goal, reference_image_path)
    return await run_full_test_bench_pipeline_execution(config, reference_image_path, quick_mode=quick_mode)


## E2E Execution & Telemetry Helpers

This cell defines the core pipeline orchestrators: `generate_audit_config_only` (Stage 1 Config) and `run_full_test_bench_pipeline_execution` (Stage 2 E2E). It coordinates retrieval, segmentation, cross-encoder reranking, and visual LLM inference, while logging telemetry and formatted scorecard widgets.

## Recall & Precision Upgrade — Code Only

This optional override layer keeps the existing AlloyDB schema unchanged. It broadens candidate retrieval, ranks all search lanes, preserves canonical asset-to-page occurrence mappings, removes irreversible drop-off gating, and expands verified canonical matches back to every matching page occurrence.

In [ ]:
# 6. Recall & Precision Upgrade — self-contained, code only
# No AlloyDB schema changes, DDL, DML, or separate file upload is required.
import base64
import zlib

_optimizer_source = zlib.decompress(base64.b64decode(
    "eNrtPe2S3LaR//cpEJ4vy7EpaiVHdmqccWWlXdlbWUmulRJfajJFcYaYGVocckyQu5qTVZVf9wBX9wz3YH6S624AJECCs7Nr+8dVRZVkOQTQ6G70NwDG87wr"
    "voiz7OG25ItUpEXOimtelmnCBVsWJavWnP0tFXWcsVMheMVO6ySt0nzFnhVZPA+Pjt6sU8E2RVJnnMFTmlc8rwAQQN2xJK7ieSz4A7FY803Mcl5XZZyFjF1U"
    "rORxIliRQz+Y5oi/TwVBvqb5ohjnE2xRZPUmF6wWPGFz6sryouLzonjH4jwBmIAxTC1qwPns7PLh2YvL8MjzvKOjZVlsWBQt66oueRSxdLMtygpGAYAYkRRH"
    "R/qd2OWLtNA/17FYZ+lc/2wbNumGV7stF/pFyeU0gGfGFwRUz5PwZVxnVZIuKtkHBiKBqvk03wXsDBoD4AYv43nGA3YJTAjYq61kYcDe1NuMN1jm9Wa7A1xZ"
    "vtWvtsADeAH/2SZylu8uLvUUF5t4BUDpz6stUHsEOMEaiQqWJypp7aNm7aNm7f1cjAm1qajKADGdjdiDr9nLIufjIwb/gL8XEgxQnvAHuIwBkxAfLNNSIGe2"
    "WbzgG5AHkouCxays8xxZoFcQBAihvX2bi7dv2aaGYXNurfGxgNZVVszjTPijt29Bdt5AswU84bBWwMGKZzuCByQJXl5LSI1gaWkEjHMQw0U1BtCWtAEOJcgp"
    "MIikk6gKNb0S1aJMVyksTZSUxbZYLtmE5WLqJbyC1dcvo2UW32RcCG9mD1rxnPCMAIVlulKDm7cxapdqi3Dy7vgs20RpvuQlzxdcjQae2u9hZIMIMAJEEPEg"
    "QLj8K15FKUpEhMIcoTT727hajxksNi0z/JWrjP+wU8AimKyR/XAFuibakaOmc8lB1XLqydKl/ItKig8hiEtZiZu0WvseIfDQGzGewYKonz9s+cprEd2WxYLm"
    "KeNcbGOkbedLzOe7ioOI0p9AK1o0X42lwkxB3AKm/2cGuPufnwRM/ndENEoIDeKwvqfZTbwTmoQYJSBj3738JmCikN2JlIQvMsAlYS8uXpyjSgOFq5LzEEVE"
    "g6vKXQsb/yHVUg/DYstzPy3Cpwjy4pVJEaAGiiyKulxwezz+k++BGK3PIYj2UrGnENyXHUa9gbASsikEQw0LkzPfu/rm6akXMO/yFBYBLL1v9phMmPedR+R6"
    "Jvc9HKs6grgVoz6SJAWreQxoqo4gzmBXKjXlyDliHi/ercqihgkVeWHObxosEWAo0v/k5lqzz5j/+MmTYOQGSWxtoMXZdh2DZoFdFGnF/XZCCX1koenAEuV0"
    "vHciB7kOOEVdbesKuhsS4FgxQlrE19yXAwJ0x5u4mnggkcCRAjzEBhgyeVPWjhVXQizHhqDx13FWc2Mi/n7BtxU7pz/o90Hu4J1N4bYE7fGX3vdxiVZ7zJZx"
    "moHkgynPEZsMEFDkY7AgTSlr7NCYfQCQHztMUKgZUt9qfLQqwMhV/H3lm6bQ6YssI0XkgUHQjnSK3VDtralNmMgU35NvcFbUBW8U3NJfYg0eclGmxDfXMI95"
    "4Q9FmvuO8fkiq8nXAgDANY0BwHRGGgh/DoYDfD0MTs80N0CBfz6xbSQXDx9RvSUr0WbI1jArbngJotOu0rbIUjDF+5YowBAKvE8jCuDLM1o3u1+7hLjqsGID"
    "ImBKLnhu6FBuBC6wR79xGUS8UdKIvyqMS5tfENRAVLqQywx+ZQlyzLaACQSG+GoZX6cwkSGqoMPE33aijK9iMIHQu8gS+lNXEFJwegZrhX8WdQn0EjqrMk5S"
    "9SyAYdRNQaXHGFYNmruUSQNss48scZzvfMQGF4mYRQG6emFwpU+DArkXgEVvCwLC2qI0mOCkCjvVpX6ih1KyZQ4SJJnBOVG94xnIEz5tirxYrCFgpQ43axBj"
    "OQIss9dFIBVRkqLSbSAMomBuYrIL5dekFX7vJdagqqcgH2wdRHjemEkxi2BUzuMySiDKAHEClFFRDEwonPH1OkeC5+Bu0mvZz8JRBj4Cwk2UzEgaz54pGSAf"
    "MBposcf/GwXLkDuVbBPvmuQO42KYXat6TnHy6woN+WO2gCWSUTakGpDe2PDqPP0RDMUizoscFQqfkhT1AAIxCNpZwxpITQBesZA6AZFcaJMG/C5TDjhE8zoB"
    "qwY0obvpG7teR4ziopMTiOQ63MKFvg2a2SdgX7jhxKtbwbRdhqDo1ILG3gLO1TdgXz45ccnXE3uyj61pntdplkDCC8lyBMtU7qQlxYQHycZUtG+pQYZysG0o"
    "ptJOg3I9h3SL9/0sxLjPIC6GxY0ZwafAFgwhA78rUC9JLB+KagcGll+j3QXzBXm7itusKBlIM+d2hQneOXq6RBl2AYIaAwUexH7eV9qTmdQd5B6PmpmkJwN6"
    "+zBAd+NysVbODsZ/+Njaii1mMjBsauG89Kg8wtCFQfDTg3nchhvHATs+Hn30OiJzxX+sU0wuVCwl0hWoNTgpcTvRB8cWM3MBHHB6MU4nzCfiw3gLeUwC0eGV"
    "9lIPZCgYQ6K6E6nocGB63AN8PDODQ0BG8lqiMWT59iHjPcNBIDs5VqmKHFM38FEP0PDW4oF2XpCegQxj7SAgz6qZTUILJg9sZbVYM8zpS5w3hxlMP91LBVxI"
    "QPghODAgiUFL6hyrAQxkA5UeiATpoZQwRteNBhOaFBpq9YrcnLOJ3n7SAoBzklOjB7DhUiaRi/BghmtYymhjCRlzq4xfF5rQHMiwunlD3Yz4DACjecdxjGat"
    "1lZGvxLjhw+7q6PQxrrRLbkxZcUIFIOHubc3EVZQVcKF5PUzm1evz8uyKIfRoQYq+6mqyFANxkf3GUmdJqvZslJKtIOVYBpwkr0Bb1MCim9iWP2hGlE7vXvm"
    "Ud+Y6QBdAgjIpvvOsaMONtOu2UNK5OORGVOAcMeJ1DHK+IC8moqfAKfOMQlDCceoAmJls5YnK3kGLFLzB2mecFQe8hYl1pKpLnyz5iq0fahiJ6maVELkPEGN"
    "CvsGZDpoPGa2PMhujcL1EkayyRhBNUGOrFFa9cVS22yC1toW07Q0Fol5/QmKJcEBXQLDhxUr9HhIott4oZnCVmIOwE8h3w1tsHbGrd0BkJelovLvkJAedcpI"
    "aAEwsm7zAKoIQSoJgzdNroijvQ47BnqiBcM3FJYrFBxlJd2kjWxn5ToEK1l2EIfyrH84x0j1NyRaeDOnXPTLCNYgqgv0BoEH/0f+nDZWUIVwWKDkphURlBvN"
    "X736JAqmbJVteBW6hOqsUOYa3E6sxBA0SoAuZHEJ2gPspzwpA17kKNktAiRZENZBqLnkMvxgSboE+7FfzjQLkx+AsAXtsERZsUoXh7CwP2iQheziOWmMEv6e"
    "4sEPt+q0JHbcsKl/Dma++fb8JRW/vmLnl6/PKWhK81rrP+4YtDtXbLHmi3dikFWyqtaN+zrbcNKQjTvILL0PysIdYzYA8RPEA016xGTeMGn6dBOn49k4+Ghk"
    "cCpvw5iiM42ihAS0B9aRqxBkb7jqJFe663P5Zg55SuNwD0hV7uh65WYA7qlNl1kRVzO5waZ+zKy85hyRoQUFdyNttq6+yLh2C7aEyZoxdJTZD6IatukEugpR"
    "FcoZ1JucosAqzTLqn8J78mYZbuI9SMVXoKOSC4neoIIRDbgbMFQVqCIWq7Hoepplxe7saWii3YZVuDOjtoTo2Wvj/EVG4i8b5Q+jtUGAUsxMdTt/8fT87Ozi"
    "5TfRi1dn55dG/20h5HJBXDyWDFXl1qmVWjgXyxI02h+aOHek9oc6e2a4PSjt0qB9itzb+g5Gh7iRGtVl6mOhEB8mrqkC1uA7wafOToR7y2Ab70D6cJ/DHZjf"
    "TriOdSQg96bEIfTJ+VDoJgrWHoJcAA+oNoym48+xOGKEEpDvN5BQZg6AYtUo5JbHdPwoegJwZ632UbJDwuw7xJMMgqH93cxAbMFtozRK/aCNsEyE0kYpeH0X"
    "Rr0mHQ0KXJ6O8Jnoh2DAGU7kIpExeib7PpNpgNzLAXZssKaI5i6tdpMvv/hjp+zk3G3R5IUNpmJ6MgtlAbDlYFYUW0xI5GEIdMyR2rGPsMm3xeGaLwJaTXho"
    "0phmLIgtBHgWMggjxO3qNI/4e76owVT6aKwDtW6BJWYdwg4YbYrWvk2QLvZdx4SzrHfzMk1UOtTSIRaQqQ5tehz1do5QhIe7He7O2jEZRHAVlRGh6RHVQWVj"
    "K+D2dLabuzKPaEgajQhCpuI5h3APQpGKnb58ybIYfm+xglwkJHSGw7uABb8u0kTmehC+XKdFjYd2NlvazH379tXV2fkVe/r3t2+x0AFyKKguc7NOF2tI5pK2"
    "LoA54HtWC1lDSmhueaCEgs90UWcVekGwmgKj5DlohDxJFGcCXsf5u9Z1Pn/z+mEVr5g6/8DmHBINzq6unsuSzzvOt7rA0NYpMRnB40Q7SkncfrbkK+AxpOQg"
    "NyCB+hCG/db0rjmk9eoEhVQReeSjimJ058kcxSSXR4e82SEq5gqbrF8HFwqsUaHglarSOqqfvU0/OdGe0oK5D4U8iUhy8RxJ/N6n56DJ13slfsNfkBfQY/UA"
    "s4rf8hqL8r2ebaG+7fiO726KMiH/gzkpFvpGEDmU6VblpHqryGYRpSqKNRqGmTDTUYsuPAu/+015LQ9oIS0QkUDHO0zaCTDuGXi0rWIdP37yhdyixENyoXzh"
    "d+CNwjV/n6QrLiqgzwrWJB60lWFVBgXHI3SRPvI3YR4dy4pSMO2rhYjK+EbFXfIwRMBavgjowjeQh5kFZtB1AoBRXB7jOSbck4J4Lgu0S46QCq+1aK0T0FIL"
    "5s8/wLa2g8mISrUPlW77VM/EH45qZnzTqDb2kG4XSG3tguOESDKHQdArTEqIiUqjc6+rBN2xUH4ydxwbKW5Eg0syD5e8Mn2fnSN6nrPh9fnl+bM37IO9mB8D"
    "Z2f1z29zoD9NvmafPBqPJZYjdvpaL0QC6McgPneC9NiCpEPIBpYT1POrVy/YB7DSx2dPo9fPvj1/cQqZdmidFHQO1L6ODZHjHHV58eLiDfvkc2cjcDkYDPDR"
    "OzgblcdwN5rWuN9j8CzRFANxHyREGit4QFuFEjNzac+yEndSHVVc1EZ1sOhvpJj4T6bhYCmA+XpnRMMY/T/QzV9D476/ePOtrIuDVQUp9wcV5D66SX4LbW/+"
    "Llok/m1dqXsRVUKZmWORbrYZPyaDu4ir6Eb4x+w4OAQO/nv26vTy/PWzc99l3Y+PR3cH1HUJ9wNSljEkqkWE7jZf+ZYjAvpGBPcQsKPDZr/hcxV3EHdJ7g3m"
    "fvLo1snIAlLchGs52Pve1o9E8dvzq/Nfvv6/bNHvv9K/1vKORuzPf77vko32udVP5foobd/vhZq1ZmdAlI6F0mSfF3p8Ny9ENLmb2rD9t3MyFO/f3cngMh7q"
    "YP7lNu7vNg5yFgr4s1d/ffnG//Qgk0kqUANzILVoVNZS0dOrq9O/T2fjMUohJEpk+6T8Q4dD5pCW7Pw/Ll6/eX0YIQYxjywUMf5TeCAauF8MryGFPxCVFh25"
    "adrQMQKt/cs5O/73Y/bTT6rRgj7C99B8kB86zH+Axq3TPR7g3u7jl5o95XoUfuxrdnKLddQ9fzPjSKI41PJbm0Z5ZhCT2/tYyG62/y9rqQTxDOzBxUt4ePWS"
    "+WYJYdQzj79ubinF25wRKPrk0X4ZN7vfIuKDUtwVhV9bZFU6im8DFbLIR9AR+SRF2VrEfbsMVt0mMBLRwIgXgr6CGNsFxvmk73m6WoM5xdJxc+5HMKxKt4eW"
    "KuMsBvtrnqXvuD78ZICiwA8PKMQykM0KPD+94VVMG7+LOGc5FipZjLsMJWfzusK3qJCbOqvSbWaewsKd+R3743t5Kw3YQ/d92pOjdNCDyvVtDoG00EF+PKJO"
    "herAXoCT8IsnwBx5TNqzFuQkfHwim+KVZ67PSfhIjmlZ6pmrFrA/hCdGOZi2+sxNkY5FAvw+tNpTlsvoHbz6orXnSzrggaX7gMkJJGUoXYpG28qQ9EEgHGgZ"
    "5Hm9oUN2vhxPe8aTR46NYq00eMGrKrH7tKlKejPHxiyeZZpIGs2KulO3mvKms/XDoJP1UGC8MVIzHIR5JETQ6yQ82deLGCmg33R2azdKJ7Dvhz3Rn9cp2ElE"
    "VSG707QnIfN61ToTUL9xANTHQwwWrtpUMWzGPptogXrIfCl/n5H8DA5UPJzpnfHBm5hGd8XL2VT+nJE7cyTmWDofZB8eMEH70N7Mdk/ZHUizoSz3Go6Mkj/Z"
    "O7JX11wcrLHmkT6pB3Kf2e8eDi7Q70v8SKDtWKK5YoI7La366dPhRvWe9kH2amZvfU/CL58gZ02bhfh21lNuUoACde5Fqn3OSZdHhJyFeO/MSjMYVg5XjSlu"
    "teh93fRp3rniL2viqTXpTPG1XUt5emDsjADtgzu4ehawoFnMLrH4/q6rqq6kKj05OszmffrpoKXzLFzxXpSFu3tMCZHadUwxjbKRdCbEt9ZhNDhY71W2plMU"
    "JcQIPgif35GgofJe16C6LMPAyG1ZbArM8OY7vQmJF9PQLcv7O/4H7b7JV39kv2d3wUyqhKlhESVe7TW4IXUZANj3B0NGCW8EDTW1hk5q5aMht+Z0HBJsv8WB"
    "88eB8zRKdHGtcY9hksWbeRKjwI+l9emKFR1YxDOo3Wva7iOg+hAGhqOC/fzP/+kfyySugPfNeO4bcdtoHHyUsZpqa8I21YIFL9WiwjbZ0J2AlvcBLq/q3kZx"
    "ckDv8KjqJ3kz2n8KVHZqb6YMfDTDT5Zjtk3CM7Aez0uq3Op7lBDOKl3Fwy8QVbanPO0Bw7/s4zB/4XxrfyVE8FUTo8NSzEHWAorE5Ydm6JQxuwQjF9OHR4CE"
    "pHupLVmaxj1ZhnyzrXbOqygmZpiV7P9tToJcT5Yj9if2BxvyGiIX4A5Mi5IaKd/bE07MBxd4HjpfTeRtP8jVt7tOpo3AQMENnkRzWHuKHzxs9FxUYUMHd5UV"
    "T7ApVD9Gh/Q5MlGBtCLBL9pk5GS632IBfliiMrKc2lLKAi0pfQPDlxAlGZjASNjeHAwqL7M05/QW5qILzQv5zRxKAL3RyFk3oTlcy01IYOMQMwmtIycfG5KN"
    "O12qYNDoYYQGHrt3NQcHd99laLHsV0O3+H+s08U7OtVoXOG/TeEMFTMvWDrvF7Wlq8wgB0MrgCi3jPxphw94Z3WVgxBHdGasY1xhKWxY91a/Ydw6E6D8tXfC"
    "hS/qOazHZNoJTgBvPHc28ej0Xffmfw/stDOLuto64JwDfWd3MmHIkJlRKvhO31fi8WJt1S1Ky+voY3JV/A5NIdpFCuDZmsdJW0fArroe04kcCQp9YKapL1ix"
    "SEdtGkg6INzLgakj9qKUK9v5yhmr1+MGEfViNAuRBv/RiVmJgEgKPRBPbHEjHgfs0wY7p8D94lWXp9KotIFC26qajG/0ATbX1fCZXUyJkLa+WBLFeNpOjgrI"
    "ZzQ0myf0ZOmyy4ema2BM89vwwkBAP97ZfRG5ktRRiBJfSRR9RK5jJOSHERqn335Ta1nn8iJjvMB7pEK6UyyxofmRNlyEjF1SZGDAwxtZIJ4xuI1E3l6kqhPH"
    "D7HBFF+B+azWajx1bu5k0cpyQ780ByLlyvXvafPgdiLaJc+6nrwBSDQcDPB3gwD1DVwT0cCepnumOuLv8Vt1Ed5IwwPlxochfHVKN+oGfuRtzBf2jR0CSMeV"
    "FQuNz1JQJQCPC8vzvJ1vUaBlwHHNV97kTnY3kGvxkj5E3Ws0Sg/k9Sk1bnqqwGU8cA5fdWu/r0Z7WgnVEnTj1ABHVl9tfJmXNDvWvrtCiB1vDpu6PvSjJp7a"
    "BM1IlfPYH4Uy1peHN1sgs+6NcDnToeTe+zj0nbeW7r6tdOCW0oHbSc6tpNu2kdoTqoP7se2ZWPfA7lnZzimYIbAHHqk9+lV2tJy7Wacv/27ulR8duK3V0j24"
    "weXc3JJyO3R5pvVb851G8EOzA2CrzIjyf9KuqNliUOqFRbISJcYffTTiPbRcuAlyWD3ONlwIzRb1tj2irSTI/nDnrX09cldWMS7v0kkGBwntAO0R3atstmBV"
    "9jt23oPCi7Pdu1d4tcPAJawKuvfljxwdw3qL4U0XP1dXM/yg9YjaArE1n7yCrNvcoIz59PoRFDS/9r0/1apjWgmgBVrnv9Dm/07bfM0mzE7pKoKR8VJirIP0"
    "TzVOENU208/szjL2a9OfFgJ5AAMghacO266/F/iqFVeaFw2KqhOpqbFQZMiLPoX083/9ty47SXSpX/PJh94HpfofSCEsXde5lnWGn5UTEELDWMg/0y3HFF9d"
    "JUNn0L+9tfeO130ub3XTaf2lJeMG12C4Y93cOn98Th/T5aV0iXnRft4Wrw0DM3GTVH52ebGO8xWyy3BI1ieo7nbz57ZvinTrnN4/8schu1Kf/i0dF9DklD//"
    "83+tjIDQ0Be4tL/tX8378DFgNm77b7KosMWGf0B1wHQNVpWi7WPDdPDh81AmDvQBD7XJD9AE82EBDbiYH+CnS+IS75rHlfxehKj4dmRzqV8UGyqqttCNi1nS"
    "SUsg6odKEYYrTMZ0Xca34r231r30/mZ+XwAA65J2i8+IfcaMVzhlx2gY7OqWsek72ATRoBpHq/oB8LRXyA7dhWtb+twfYvaH+HiIYE5uu3dlY3BAGtUziur9"
    "HRPqgOVxRJ/LBYM28bIYE/ahvJqmzIX2zUfu/TzPcdPfG7vu/3e+i+b6FDQMdL3ujHTfLseNMmdDZ7TjqiUMdbwNunQOftF7vOdTUx0oPVuHByC67zpjhj5E"
    "Ph4yDI45b/eVCpHbO5pfKWxVS9lD9QV5niin4Pq/IQjZmfZrvU+z0+czbB9HlpPOHICR/D8tYg/e"
)).decode("utf-8")
_optimizer_namespace = {}
exec(compile(_optimizer_source, "<recall_precision_upgrade>", "exec"), _optimizer_namespace)
_optimizer_namespace["install_recall_precision_overrides"](globals())


In [ ]:
# TEST RUN (Stage 1): Generate Audit Rules & Configuration
# Run this cell to upload your reference image and generate the rule configuration.
import nest_asyncio
nest_asyncio.apply()

# Dynamic Reference Image Detector (Triggered via Colab Interactive Upload)
reference_image_path = None
try:
    from google.colab import files
    print("[OPTIONAL] Upload a reference image for comparative compliance audit:")
    uploaded = files.upload()
    if uploaded:
        reference_image_path = list(uploaded.keys())[0]
        print(f" Reference image uploaded: {reference_image_path}")
    else:
        print(" No reference image uploaded. Running text-only audit goal.")
except Exception as e:
    reference_image_path = None

# Enterprise Audit Goal (Modify this as needed)
TEST_AUDIT_GOAL = "Find all pages with this exact image"

# Step 1: Generate configuration rules
audit_config_global = None
try:
    audit_config_global = await generate_audit_config_only(
        user_goal=TEST_AUDIT_GOAL,
        reference_image_path=reference_image_path
    )
    print("💡 TIP: You can inspect and tweak 'audit_config_global' directly in the cell below before running Stage 2.")
except Exception as e:
    print(f"Error generating audit configuration: {e}")


In [ ]:
# TEST RUN (Stage 2): Execute Visual Search & Parallel Audit
# Run this cell to execute retrieval, segmentation, and LLM inference.
# You can uncomment and modify rules below to calibrate config before executing.

if 'audit_config_global' in globals() and audit_config_global is not None:
    # OPTIONAL CALIBRATION TUNING:
    # If the generated AI rules were slightly off, you can uncomment and edit them here:
    # audit_config_global["inclusion_criteria"] = [
    #     "The image must contain the legacy Google Pay logo featuring the interlocking loops design."
    # ]
    # audit_config_global["exclusion_criteria"] = [
    #     "Exclude images containing the current compliant Google Pay GPay wordmark button logo."
    # ]

    df_results_global = None
    try:
        df_results_global = await run_full_test_bench_pipeline_execution(
            audit_config=audit_config_global,
            reference_image_path=reference_image_path
        )
    except Exception as e:
        print(f"Error executing test bench pipeline: {e}")
else:
    print(" Please run Stage 1 cell first to generate 'audit_config_global'.")
